# Classical Methods Lab — Group 14: Read Stripe Direction

**Pipeline:** Stage A (Owner A) → Stage B (Owner B) → Stage C (Owner C)  
Uses only NumPy, OpenCV, matplotlib — no deep learning.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

# Project modules
import config
from stage_a import run_stage_a
from stage_b import run_stage_b
from stage_c import run_stage_c
from run_pipeline import (
    load_grayscale_swatches,
    write_csv,
    write_results_markdown,
    save_labeled_strip,
    save_trial_a_plot,
    save_trial_b_plot,
    save_trial_c_plot,
    save_gradient_maps,
)

config.OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print('Config loaded. KSIZE =', config.KSIZE, '| BORDER =', config.BORDER, '| MARGINS =', config.MARGINS)

## Load Grayscale Swatches
If `images/` does not contain the real swatch files, run `make_test_swatches.py` first.

In [ ]:
images = load_grayscale_swatches(config.IMAGES_DIR, config.SWATCH_FILES)

fig, axes = plt.subplots(1, len(images), figsize=(12, 4))
for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(name)
    ax.axis('off')
plt.suptitle('Input Swatches (fixed scale 0..255)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Stage A — Owner A: Measure Change Across Columns (Sobel Gx)

**Key idea:** Vertical stripes produce large brightness changes *across columns* → strong horizontal gradient `Gx`.  
We use `CV_64F` so dark-to-bright (positive) and bright-to-dark (negative) edges both contribute.  
**Comparison:** mean signed Gx (can cancel to ~0) vs mean |Gx| (always large if edges exist).

In [ ]:
# STAGE A (OWNER: A)
handoff_gx, trial_a_rows, abs_gx_maps = run_stage_a(images, ksize=config.KSIZE)

print('=== Stage A Handoff to C: mean absolute |Gx| per swatch ===')
for name, val in handoff_gx.items():
    print(f'  {name}: {val:.4f}')

print('\n=== Stage A Trial: Signed vs Absolute Gx ===')
print(f'{"Swatch":<14} | {"Signed Gx":>12} | {"Abs |Gx|":>12}')
print('-' * 44)
for r in trial_a_rows:
    print(f"{r['swatch']:<14} | {r['mean_signed_Gx']:>12.4f} | {r['mean_abs_Gx']:>12.4f}")

In [ ]:
# Visualise Stage A comparison figure
save_trial_a_plot(trial_a_rows, config.OUTPUTS_DIR / 'trial_A_signed_vs_abs.png')
img_trial_a = plt.imread(str(config.OUTPUTS_DIR / 'trial_A_signed_vs_abs.png'))
plt.figure(figsize=(8, 5)); plt.imshow(img_trial_a); plt.axis('off'); plt.show()

---
## Stage B — Owner B: Measure Change Across Rows (Sobel Gy)

**Key idea:** Horizontal stripes produce large brightness changes *across rows* → strong vertical gradient `Gy`.  
Uses the **same `KSIZE`** as Stage A (shared from `config.py`).  
**Comparison:** full-image |Gy| vs border-cropped |Gy| (5-px border excluded).

In [ ]:
# STAGE B (OWNER: B)
handoff_gy, trial_b_rows, abs_gy_maps = run_stage_b(images, ksize=config.KSIZE, border=config.BORDER)

print('=== Stage B Handoff to C: mean absolute |Gy| per swatch ===')
for name, val in handoff_gy.items():
    print(f'  {name}: {val:.4f}')

print('\n=== Stage B Trial: Full-Image vs No-Border |Gy| ===')
print(f'{"Swatch":<14} | {"Full |Gy|":>10} | {"No-Border":>10} | {"Diff":>8}')
print('-' * 54)
for r in trial_b_rows:
    print(f"{r['swatch']:<14} | {r['mean_abs_Gy_full']:>10.4f} | "
          f"{r['mean_abs_Gy_no_border']:>10.4f} | {r['difference_full_minus_cropped']:>8.4f}")

In [ ]:
# Visualise Stage B comparison figure
save_trial_b_plot(trial_b_rows, config.OUTPUTS_DIR / 'trial_B_border.png')
img_trial_b = plt.imread(str(config.OUTPUTS_DIR / 'trial_B_border.png'))
plt.figure(figsize=(8, 5)); plt.imshow(img_trial_b); plt.axis('off'); plt.show()

---
## Stage C — Owner C: Decide Stripe Direction

**Rule:** Given handoff scores from A and B:  
- `Gx ≥ margin × Gy` → **vertical**  
- `Gy ≥ margin × Gx` → **horizontal**  
- Otherwise → **uncertain**

Tested with margin **1.0** (any dominance) and **2.5** (clear dominance required).  
Main reported decision uses margin **2.5**.

In [ ]:
# STAGE C (OWNER: C)
results_rows, trial_c_rows, uncertain_cases = run_stage_c(
    handoff_gx, handoff_gy,
    margins=config.MARGINS,
    main_margin=config.MAIN_MARGIN,
    comparison_margin=config.COMPARISON_MARGIN,
)

print('=== Final Results (Stage C Decisions) ===')
print(f'{"Swatch":<14} | {"mean|Gx|":>9} | {"mean|Gy|":>9} | {"m=2.5":^12} | {"m=1.0":^12}')
print('-' * 68)
for r in results_rows:
    print(f"{r['swatch']:<14} | {r['mean_abs_Gx']:>9.4f} | {r['mean_abs_Gy']:>9.4f} | "
          f"{r[f'decision (margin {config.MAIN_MARGIN})']:^12} | "
          f"{r[f'decision (margin {config.COMPARISON_MARGIN})']:^12}")
print()
for m in config.MARGINS:
    u = uncertain_cases[m]
    print(f'Uncertain at margin {m}: {u if u else "None"}')

In [ ]:
# Visualise Stage C margin comparison
save_trial_c_plot(trial_c_rows, config.OUTPUTS_DIR / 'trial_C_margins.png')
img_trial_c = plt.imread(str(config.OUTPUTS_DIR / 'trial_C_margins.png'))
plt.figure(figsize=(8, 5)); plt.imshow(img_trial_c); plt.axis('off'); plt.show()

---
## Outputs: Labeled Strip & Gradient Maps

In [ ]:
# Save labeled strip and show inline
save_labeled_strip(images, results_rows, config.OUTPUTS_DIR / 'labeled_strip.png')
strip_img = plt.imread(str(config.OUTPUTS_DIR / 'labeled_strip.png'))
plt.figure(figsize=(12, 4)); plt.imshow(strip_img); plt.axis('off')
plt.title('labeled_strip.png — final deliverable'); plt.show()

In [ ]:
# Save |Gx| and |Gy| gradient maps and show inline
save_gradient_maps(abs_gx_maps, abs_gy_maps, config.OUTPUTS_DIR / 'gradient_maps.png')
gmap_img = plt.imread(str(config.OUTPUTS_DIR / 'gradient_maps.png'))
plt.figure(figsize=(12, 7)); plt.imshow(gmap_img); plt.axis('off')
plt.title('gradient_maps.png — shared display scale'); plt.show()

---
## Save All CSVs and Handoffs JSON

In [ ]:
import json

# handoffs.json
handoffs_payload = {
    'shared_ksize': config.KSIZE,
    'stage_a_to_c_mean_abs_gx': handoff_gx,
    'stage_b_to_c_mean_abs_gy': handoff_gy,
}
with open(config.OUTPUTS_DIR / 'handoffs.json', 'w') as f:
    json.dump(handoffs_payload, f, indent=2)
print('handoffs.json saved'); print(json.dumps(handoffs_payload, indent=2))

# results_table.csv and .md
results_fields = [
    'swatch', 'mean_abs_Gx', 'mean_abs_Gy',
    f'decision (margin {config.MAIN_MARGIN})',
    f'decision (margin {config.COMPARISON_MARGIN})',
]
write_csv(config.OUTPUTS_DIR / 'results_table.csv', results_fields, results_rows)
write_results_markdown(config.OUTPUTS_DIR / 'results_table.md', results_rows)
print('results_table.csv and results_table.md saved')

# Trial CSVs
write_csv(config.OUTPUTS_DIR / 'trial_A_signed_vs_abs.csv', ['swatch','mean_signed_Gx','mean_abs_Gx'], trial_a_rows)
write_csv(config.OUTPUTS_DIR / 'trial_B_border.csv',
          ['swatch','mean_abs_Gy_full','mean_abs_Gy_no_border','difference_full_minus_cropped'], trial_b_rows)
write_csv(config.OUTPUTS_DIR / 'trial_C_margins.csv',
          ['swatch','mean_abs_Gx','mean_abs_Gy','dominant_to_weaker_ratio',
           'decision_margin_1.0','decision_margin_2.5','is_uncertain_at_2.5'], trial_c_rows)
print('All trial CSVs saved.')
print(f'All outputs in: {config.OUTPUTS_DIR}')